## Cohort Analysis

### Import Packages

In [57]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

### Loading the Data

In [58]:
df = pd.read_csv('https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/cohort/cohort_analysis.csv',
                 parse_dates=['acquisition_date', 'cancellation_month']
                 )
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


#### First Look at the Data

In [59]:
print(df.shape)
df.info()

(12000, 12)
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   user_id             12000 non-null  int64         
 1   acquisition_date    12000 non-null  datetime64[us]
 2   cancellation_month  7066 non-null   datetime64[us]
 3   gender              12000 non-null  str           
 4   marital_status      12000 non-null  str           
 5   age                 12000 non-null  int64         
 6   income_segment      11369 non-null  str           
 7   country             12000 non-null  str           
 8   channel             12000 non-null  str           
 9   campaign_id         12000 non-null  str           
 10  device_type         12000 non-null  str           
 11  plan_type           12000 non-null  str           
dtypes: datetime64[us](2), int64(2), str(8)
memory usage: 1.7 MB


#### Cohort Month

In [60]:
df["acquisition_month"] = df["acquisition_date"].dt.to_period("M")
df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


#### Explicit Churn Flag

In [61]:
df["is_churned"] = df["cancellation_month"].notna().astype(int)
print(df["is_churned"].value_counts())

is_churned
1    7066
0    4934
Name: count, dtype: int64


#### Calculate Month of Churn/Tenure

In [62]:
df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
df["cancellation_month"] = pd.to_datetime(df["cancellation_month"], errors="coerce")

df["month_churned"] = np.where(
    df["cancellation_month"].notna(),
    ((df["cancellation_month"].dt.year - df["acquisition_date"].dt.year) *12 +
     (df["cancellation_month"].dt.month - df["acquisition_date"].dt.month)),
    np.nan
)

In [63]:
df[["user_id", "acquisition_date", "acquisition_month","cancellation_month", "month_churned"]].head(10)

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-02-01,7.0
1,2,2024-04-01,2024-04,2024-05-01,1.0
2,3,2024-05-01,2024-05,2024-07-01,2.0
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0
5,6,2024-08-01,2024-08,NaT,NaN
6,7,2024-05-01,2024-05,NaT,NaN
7,8,2024-05-01,2024-05,2024-06-01,1.0
8,9,2024-07-01,2024-07,NaT,NaN
9,10,2024-02-01,2024-02,2024-12-01,10.0


#### Data Preperation | Keep Only Valid Churn Months


In [64]:
df["month_churned"] = df["month_churned"].where(
    df["month_churned"].between(1, 12),
    np.nan
    
)

df["month_churned"]. value_counts(dropna=False).sort_index()

month_churned
1.0     2392
2.0     1363
3.0      845
4.0      585
5.0      443
6.0      362
7.0      266
8.0      248
9.0      182
10.0     168
11.0     113
12.0      99
NaN     4934
Name: count, dtype: int64

## Homework

In [65]:
import pandas as pd
import numpy as np
import plotly.express as px

def plot_churn_by_feature(feature_name, title_arm):
    feature_churn = (
        df[df["month_churned"].notna()]
        .groupby([feature_name, "month_churned"])
        .agg(churned_users=("user_id", "count"))
        .reset_index()
    )
    
    feature_base = (
        df.groupby(feature_name)
        .agg(total_users=("user_id", "count"))
        .reset_index()
    )
    
    feature_data = feature_churn.merge(feature_base, on=feature_name, how="left")
    feature_data["churn_rate"] = feature_data["churned_users"] / feature_data["total_users"]
    
    fig = px.line(
        feature_data,
        x="month_churned",
        y="churn_rate",
        color=feature_name,
        markers=True,
        title=f"Ամսական հեռացման գործակիցն ըստ {title_arm}-ի",
        labels={
            "month_churned": "Ամիսներ ներգրավումից հետո",
            "churn_rate": "Հեռացման գործակից (Churn Rate)",
            feature_name: feature_name.capitalize()
        }
    )
    
    fig.update_layout(
        yaxis_tickformat=".0%",
        xaxis=dict(dtick=1),
        hovermode="x unified"
    )
    
    fig.show()

    

#### Marital Status

In [66]:
plot_churn_by_feature('marital_status', 'ընտանեկան կարգավիճակի')

#### Income Segment

In [67]:
plot_churn_by_feature("income_segment", "եկամտի սեգմենտի")

#### Country

In [68]:
plot_churn_by_feature("country", "երկրի")

#### Channel

In [69]:
plot_churn_by_feature("channel", "ներգրավման անցուղու (Channel)")

#### Campaign

In [70]:
plot_churn_by_feature("campaign_id", "մարքեթինգային արշավի (Campaign ID)")

### Cohort Retention Heatmap

In [71]:
heatmap_data = cohort_data.pivot(
    index="acquisition_month",
    columns="month_churned",
    values="retention_rate",
)

heatmap_data

NameError: name 'cohort_data' is not defined

In [ ]:
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Blues",
    text_auto=".0%",
    labels={
        "x": "Months Since Acquisition",
        "y": "Acquisition Month",
        "color": "Retention Rate",
    },
    title="Cohort Retention Heatmap"
)



fig.update_xaxes(
    tickmode="array",
    tickvals=list(heatmap_data.columns),
    ticktext=[str(x) for x in heatmap_data.columns],
    side="top"
)

fig.update_yaxes(
    tickmode="array",
    tickvals=list(heatmap_data.index),
    ticktext=[str(y) for y in heatmap_data.index],
    autorange="reversed"
)

fig.show()

NameError: name 'heatmap_data' is not defined